**Выполнил: Александр Рябков, P3123**

# Лабораторная работа 8. Скрапинг

##  Цель работы
Освоение методов скрапинга данных из веб-страниц.

## Теоретические сведения
Основные понятия:

Скрейпинг / Веб-скрейпинг — это технология получения веб-данных путём извлечения их со страниц веб-ресурсов.


Объект для скрапинга: https://news.itmo.ru/ru.

Необходимо сохранить полученные данных в формате csv внутри колаба.

Структура данных :

1. Идентификатор новости (целочисленное число из URL)
2. Название новости
3. Дата её размещения
3. URL на страницу с конкретной новостью.

In [5]:
import csv
import requests
from bs4 import BeautifulSoup


In [6]:
DOMAIN = 'https://news.itmo.ru'
SEARCH_DOMAIN = 'https://news.itmo.ru/ru/search/?search='

In [ ]:
headers = []
urls = []

queries = ['нейротехнологии', 'нейротехнологии и программирование']

# div.weeklyevents > h2 > span - text
# li.weeklyevent > h4 > a - text

"""
<li class="weeklyevent"><h4><a href="/ru/education/cooperation/news/14792/">Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО</a></h4><p>… Native»
	«Компьютерные системы и технологии»
	«Разработка программного обеспечения»
	«Инженерия искусственного интеллекта»
	«Языковые модели и искусственный интеллект»
	«<span class="selection">Нейротехнологии и программирование</span>»
	«Системное и прикладное программное обеспечение»
|s=2,center|
«Летающая робототехника» ― 49 финалистов. Партнеры ― «Клевер COEX», «Свеза» и Акаде…</p><p>06.04.2026</p></li>
"""


for query in queries:
  data = requests.get(SEARCH_DOMAIN+query)
  # Парсинг HTML
  bs = BeautifulSoup(data.text, 'html.parser')
  
  # Подсчёт новостей
  total_news = bs.find('div', {'class': 'weeklyevents'})
  total_news_count = int(total_news.find('h2').find('span').get_text().split()[0])
  
  # Сбор данных о каждой новости
  # извлекает текст заголовка и относительную ссылку (/ru/education/...), к которой прибавляет домен, получая полный URL
  news_headers = bs.find_all('li', {'class': 'weeklyevent'})
  for _n in news_headers:
    post = _n.find('h4').find('a').get_text()
    headers.append(post)
    post_url = _n.find('h4').find('a').get('href')
    urls.append(DOMAIN+post_url)

print(len(urls))


20


In [ ]:
urls
# 20 записей (часть из них дублируется, т r оба запроса находят одни и те же новости)

['https://news.itmo.ru/ru/education/cooperation/news/14792/',
 'https://news.itmo.ru/ru/education/trend/news/14771/',
 'https://news.itmo.ru/ru/education/official/news/14666/',
 'https://news.itmo.ru/ru/education/trend/news/14508/',
 'https://news.itmo.ru/ru/education/official/news/14481/',
 'https://news.itmo.ru/ru/university_live/ratings/news/14438/',
 'https://news.itmo.ru/ru/education/trend/news/14434/',
 'https://news.itmo.ru/ru/education/cooperation/news/14405/',
 'https://news.itmo.ru/ru/university_live/achievements/news/14290/',
 'https://news.itmo.ru/ru/education/trend/news/13699/',
 'https://news.itmo.ru/ru/education/cooperation/news/14792/',
 'https://news.itmo.ru/ru/education/trend/news/14771/',
 'https://news.itmo.ru/ru/education/official/news/14666/',
 'https://news.itmo.ru/ru/education/trend/news/14508/',
 'https://news.itmo.ru/ru/education/official/news/14481/',
 'https://news.itmo.ru/ru/university_live/ratings/news/14438/',
 'https://news.itmo.ru/ru/education/trend/new

In [22]:
print(headers[:2])
print(urls[:2])

['Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО', 'Топ образование по программированию и ИИ в 2026-м: на что смотреть и как выбирать']
['https://news.itmo.ru/ru/education/cooperation/news/14792/', 'https://news.itmo.ru/ru/education/trend/news/14771/']


In [25]:
import csv
import requests
from bs4 import BeautifulSoup
import random
import time
import re # для извлечения ID из URL

# Списки `headers` и `urls` уже заполнены из предыдущих ячеек.
#  зайти на страницу каждой найденной новости, вытащить дату публикации и сохранить все в CSV-файл.

news_data = []
output_csv_file = 'scraped_news_data.csv'

print("Начинается сбор даты для каждой новости. Это может занять некоторое время...")

for i, (header, url) in enumerate(zip(headers, urls)):
    news_id = 'N/A'
    news_date = 'Дата не найдена'

    # Извлечение ID из URL
    match = re.search(r'/(\d+)/?$', url)
    if match:
        news_id = match.group(1)

    try:
        # Заход на страницу каждой новости
        time.sleep(random.randint(0, 1))
        response = requests.get(url)
        response.raise_for_status() # Вызывает исключение для HTTP ошибок
        soup_news = BeautifulSoup(response.text, 'html.parser')

        # Поиск даты размещения новости на странице
        # На основе ручного анализа, дата находится в <div class="time"><time>...
        
        # На странице новости ищет блок с датой публикации и извлекает текст из него.
        date_element = soup_news.find('div', class_='time')
        if date_element and date_element.find('time'):
            news_date = date_element.find('time').get_text(strip=True)
        # else:
        #    print(f"Предупреждение: Дата не найдена для {url}")

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе {url}: {e}")
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")

    # Сохранение данных
    news_data.append({
        'Идентификатор новости': news_id,
        'Название новости': header,
        'Дата её размещения': news_date,
        'URL на страницу с конкретной новостью': url
    })

    if (i + 1) % 10 == 0:
        print(f"Обработано {i + 1} новостей...")

print("Сбор данных завершен. Начинается запись в CSV.")

# Определение заголовков для DictWriter
fieldnames = [
    'Идентификатор новости',
    'Название новости',
    'Дата её размещения',
    'URL на страницу с конкретной новостью'
]

# Запись данных в CSV с использованием DictWriter
try:


    with open(output_csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(news_data)
    print(f"Данные успешно записаны в файл {output_csv_file}")
except IOError as e:
    print(f"Ошибка при записи в файл {output_csv_file}: {e}")


Начинается сбор даты для каждой новости. Это может занять некоторое время...
Обработано 10 новостей...
Обработано 20 новостей...
Сбор данных завершен. Начинается запись в CSV.
Данные успешно записаны в файл scraped_news_data.csv


In [27]:
# найти сколько всего страниц в разделе главных новостей (/ru/main_news/).
# На каждой странице смотрит на кнопку навигации "следующая страница". Пока страницы есть — собираем данные

URL = 'https://news.itmo.ru/ru/main_news/'

start_page = 1
max_page = 200
total_pages = None

for i in range(1, 200, 1):
  time.sleep(random.randint(0, 1))
  data = requests.get(URL + str(i)+ '/')

  print(data.url)

  page = BeautifulSoup(data.text, 'html.parser')

  next_page_href = page.find_all('div', {'class': 'pagination'})[0]\
                                .find('ul')\
                                .find_all('li')[1]\
                                .find('a')['href']

  if next_page_href == '#':
    total_pages = i
    break


print(total_pages)


# for _n in news_headers:
#     post = _n.find('h4').find('a').get_text()
#     headers.append(post)
#     post_url = _n.find('h4').find('a')['href']

https://news.itmo.ru/ru/main_news/1/
https://news.itmo.ru/ru/main_news/2/
https://news.itmo.ru/ru/main_news/3/
https://news.itmo.ru/ru/main_news/4/
https://news.itmo.ru/ru/main_news/5/
https://news.itmo.ru/ru/main_news/6/
https://news.itmo.ru/ru/main_news/7/
https://news.itmo.ru/ru/main_news/8/
https://news.itmo.ru/ru/main_news/9/
https://news.itmo.ru/ru/main_news/10/
https://news.itmo.ru/ru/main_news/11/
https://news.itmo.ru/ru/main_news/12/
https://news.itmo.ru/ru/main_news/13/
https://news.itmo.ru/ru/main_news/14/
https://news.itmo.ru/ru/main_news/15/
https://news.itmo.ru/ru/main_news/16/
https://news.itmo.ru/ru/main_news/17/
https://news.itmo.ru/ru/main_news/18/
https://news.itmo.ru/ru/main_news/19/
https://news.itmo.ru/ru/main_news/20/
https://news.itmo.ru/ru/main_news/21/
https://news.itmo.ru/ru/main_news/22/
https://news.itmo.ru/ru/main_news/23/
https://news.itmo.ru/ru/main_news/24/
https://news.itmo.ru/ru/main_news/25/
https://news.itmo.ru/ru/main_news/26/
https://news.itmo.ru/

ConnectTimeout: HTTPSConnectionPool(host='news.itmo.ru', port=443): Max retries exceeded with url: /ru/main_news/35/ (Caused by ConnectTimeoutError(<HTTPSConnection(host='news.itmo.ru', port=443) at 0x1ee2f7eb750>, 'Connection to news.itmo.ru timed out. (connect timeout=None)'))